[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-2/lab-2.1-lowering-ladder.ipynb)

# LAB·2.1 · The lowering ladder, traced

**Hardware:** any machine. Both IRs you read here are backend-independent.

One function, read at every layer it passes through. The goal is fluency, not novelty: by the end you should be able to point at any line of the StableHLO and name the source line it came from. The site's EX·05 instrument is this lab, frozen for attention.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
def f(x, w, b):
    h = x @ w + b
    return jax.nn.relu(h).mean(axis=-1)

args = [jax.ShapeDtypeStruct(s, jnp.bfloat16) for s in [(32, 64), (64, 128), (128,)]]

print(jax.make_jaxpr(f)(*args))

In [ ]:
print(jax.jit(f).lower(*args).as_text())

## Read them against each other

Three things to find and write down:
1. Where the `+ b` broadcast became explicit (`broadcast_in_dim`): shapes that Python hid, the IR states.
2. What dtype the `mean` accumulates in. Compare with the attention dump on the site's EX·05: JAX upcasts bf16 sums to f32, then converts back. Find the same pair of `convert` ops here.
3. The `dot_general` dimension_numbers. Decode them by hand: which axes contract, which batch. Do this until it is boring; the recognizer skill you need later is exactly this decoding at a glance.

## The optimized layer

`jax.jit(f).lower(*args).compile().as_text()` prints the *optimized* HLO for the backend you are on. On CPU it tells you about your CPU; run this cell on the TPU runtime and the interesting part appears: `fusion` ops with their operands, which are XLA's actual decisions. LAB·2.2 hunts inside them.

In [ ]:
print(jax.jit(f).lower(*args).compile().as_text()[:2000])